In [1]:
import numpy as np
import pandas as pd
import scipy
import scipy.stats
import matplotlib as mpl   
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns
import arviz as az
import pymc as pm
%matplotlib inline

In [2]:
import fastf1
import pandas as pd

# 2025年已完成的22场比赛
completed_races = [
    ('Australia', 1, '2025-03-16'),
    ('China', 2, '2025-03-23'),
    ('Japan', 3, '2025-04-06'),
    ('Bahrain', 4, '2025-04-13'),
    ('Saudi Arabia', 5, '2025-04-20'),
    ('Miami', 6, '2025-05-04'),
    ('Emilia Romagna', 7, '2025-05-18'),
    ('Monaco', 8, '2025-05-25'),
    ('Spain', 9, '2025-06-01'),
    ('Canada', 10, '2025-06-15'),
    ('Austria', 11, '2025-06-29'),
    ('Great Britain', 12, '2025-07-06'),
    ('Belgium', 13, '2025-07-27'),
    ('Hungary', 14, '2025-08-03'),
    ('Netherlands', 15, '2025-08-31'),
    ('Italy', 16, '2025-09-07'),
    ('Azerbaijan', 17, '2025-09-21'),
    ('Singapore', 18, '2025-10-05'),
    ('United States', 19, '2025-10-19'),
    ('Mexico', 20, '2025-10-26'),
    ('Brazil', 21, '2025-11-09'),
    ('Las Vegas', 22, '2025-11-22'),
    ('Qatar', 23, '2025-11-30'),
]

# not happend yet games
future_races = [
    ('Qatar', 23, '2025-11-30'),
    ('Abu Dhabi', 24, '2025-12-07'),
]

# track types
track_types = {
    'Australia': 'balanced',
    'China': 'high_speed',
    'Japan': 'technical',
    'Bahrain': 'high_speed',
    'Saudi Arabia': 'high_speed',
    'Miami': 'high_speed',
    'Emilia Romagna': 'technical',
    'Monaco': 'technical',
    'Spain': 'balanced',
    'Canada': 'balanced',
    'Austria': 'technical',
    'Great Britain': 'balanced',
    'Belgium': 'high_speed',
    'Hungary': 'technical',
    'Netherlands': 'technical',
    'Italy': 'high_speed',
    'Azerbaijan': 'high_speed',
    'Singapore': 'technical',
    'United States': 'balanced',
    'Mexico': 'balanced',
    'Brazil': 'balanced',
    'Las Vegas': 'high_speed',
    'Qatar': 'high_speed',
    'Abu Dhabi': 'balanced',
}

In [3]:
import fastf1
import pandas as pd
import numpy as np
from datetime import datetime
import os

# 设置缓存
cache_dir = './f1_cache'
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)
    print(f"✅ Created cache directory: {cache_dir}")

fastf1.Cache.enable_cache(cache_dir)

# 赛道类型映射
track_types = {
    # 2024 赛季
    'Bahrain': 'high_speed', 'Saudi Arabia': 'high_speed', 'Australia': 'balanced',
    'Japan': 'technical', 'China': 'high_speed', 'Miami': 'high_speed',
    'Emilia Romagna': 'technical', 'Monaco': 'technical', 'Canada': 'balanced',
    'Spain': 'balanced', 'Austria': 'technical', 'Great Britain': 'balanced',
    'Hungary': 'technical', 'Belgium': 'high_speed', 'Netherlands': 'technical',
    'Italy': 'high_speed', 'Azerbaijan': 'high_speed', 'Singapore': 'technical',
    'United States': 'balanced', 'Mexico': 'balanced', 'Brazil': 'balanced',
    'Las Vegas': 'high_speed', 'Qatar': 'high_speed', 'Abu Dhabi': 'balanced',
    # 2025 赛季（如有新增赛道）
}

import os

def is_race_cached(year, race_name, session_type):
    """
    检查特定比赛session是否已在cache中
    """
    cache_dir = './f1_cache'
    # FastF1的cache文件命名模式
    cache_pattern = f"{year}*{race_name.replace(' ', '_')}*{session_type}*"
    
    if not os.path.exists(cache_dir):
        return False
    
    # 检查是否有匹配的cache文件
    for root, dirs, files in os.walk(cache_dir):
        if any(race_name.replace(' ', '_') in d and str(year) in d for d in dirs):
            return True
    return False


def fetch_race_results(year, race_name, round_num, date):
    """
    获取单场比赛数据（包括排位赛和正赛）
    """
    try:
        # 检查cache状态
        race_cached = is_race_cached(year, race_name, 'R')
        quali_cached = is_race_cached(year, race_name, 'Q')
        
        if race_cached and quali_cached:
            print(f"✓ Round {round_num}: {race_name} ({year}) - Using cached data")
        else:
            print(f"⬇ Round {round_num}: {race_name} ({year}) - Downloading data...")
        
        # ========== 1. 获取排位赛数据 ==========
        try:
            quali_session = fastf1.get_session(year, race_name, 'Q')
            quali_session.load()
            quali_results = quali_session.results
            
            # 提取排位赛位置
            qualifying_positions = dict(zip(
                quali_results['Abbreviation'],
                quali_results['Position']
            ))
            
        except Exception as e:
            print(f"  ⚠️ Could not load qualifying: {e}")
            qualifying_positions = {}
        
        # ========== 2. 获取正赛数据 ==========
        race_session = fastf1.get_session(year, race_name, 'R')
        race_session.load()
        results = race_session.results
        
        # ========== 3. 合并数据 ==========
        race_data = pd.DataFrame({
            'Round': round_num,
            'Race': race_name,
            'Date': date,
            'TrackType': track_types.get(race_name, 'unknown'),
            'Driver': results['BroadcastName'],
            'DriverCode': results['Abbreviation'],
            'DriverNumber': results['DriverNumber'],
            'Team': results['TeamName'],
            'GridPosition': results['GridPosition'],
            'Position': results['Position'],
            'Points': results['Points'],
            'Status': results['Status'],
            'Time': results['Time'],
        })
        
        # ========== 4. 添加排位赛成绩 ==========
        race_data['QualifyingPosition'] = race_data['DriverCode'].map(qualifying_positions)
        
        # 如果排位赛数据缺失，用 GridPosition 填充
        race_data['QualifyingPosition'] = race_data['QualifyingPosition'].fillna(
            race_data['GridPosition']
        )
        
        return race_data
        
    except Exception as e:
        print(f"❌ Error loading {race_name}: {e}")
        return None


In [ ]:

def collect_race_data(years=[2025], force_refresh=False):
    """
    收集多个赛季的比赛数据
    
    Parameters:
    -----------
    years : list
        要收集的年份列表，例如 [2024, 2025]
    force_refresh : bool
        是否强制刷新数据（忽略已有CSV）
    """
    csv_file = './data/f1_multi_season_results.csv'
    
    # 检查CSV是否已存在
    if os.path.exists(csv_file) and not force_refresh:
        print(f"✅ Found existing data file: {csv_file}")
        print("Loading...")
        full_data = pd.read_csv(csv_file)
        print(f"Loaded {len(full_data)} records from {full_data['Round'].nunique()} races")
        return full_data
    
    print(f"Starting data collection for years: {years}...")
    all_races = []
    
    for year in years:
        print(f"\n{'='*60}")
        print(f"COLLECTING {year} SEASON")
        print(f"{'='*60}")
        
        # 获取该年份的所有比赛
        try:
            schedule = fastf1.get_event_schedule(year)
            
            # 只处理已完成的比赛
            completed = schedule[schedule['EventDate'] < pd.Timestamp.now()]
            
            for idx, event in completed.iterrows():
                round_num = event['RoundNumber']
                race_name = event['EventName']
                date = event['EventDate'].strftime('%Y-%m-%d')
                
                race_data = fetch_race_results(year, race_name, round_num, date)
                
                if race_data is not None:
                    # 添加赛季年份列
                    race_data['Season'] = year
                    all_races.append(race_data)
                    
        except Exception as e:
            print(f"❌ Error getting schedule for {year}: {e}")
    
    # 合并所有数据
    if all_races:
        full_data = pd.concat(all_races, ignore_index=True)
        
        # 重新编号 Round（跨年份连续）
        full_data = full_data.sort_values(['Season', 'Round'])
        # full_data['GlobalRound'] = range(1, len(full_data.groupby(['Season', 'Round']).ngroups) + 1)
        full_data['GlobalRound'] = full_data.groupby(['Season', 'Round']).ngroup() + 1
        
        # 保存为CSV
        full_data.to_csv(csv_file, index=False)
        
        print(f"\n{'='*60}")
        print(f"DATA COLLECTION COMPLETE")
        print(f"{'='*60}")
        print(f"✓ Total records: {len(full_data)}")
        print(f"✓ Seasons: {full_data['Season'].unique()}")
        print(f"✓ Total races: {full_data.groupby(['Season', 'Round']).ngroups}")
        print(f"✓ Unique drivers: {full_data['DriverCode'].nunique()}")
        
        # 显示每个赛季的数据量
        print("\nRecords per season:")
        print(full_data['Season'].value_counts().sort_index())
        
        return full_data
    else:
        print("❌ No data collected")
        return None

In [5]:
def extract_driver_info(full_season):
    """
    从比赛数据中提取车手信息
    """
    # 获取最新一场比赛的车手信息
    latest_race = full_season[full_season['Round'] == full_season['Round'].max()]
    
    # ✅ 修改：只提取存在的列
    driver_info = latest_race[['Driver', 'DriverCode', 'Team']].copy()
    driver_info = driver_info.drop_duplicates(subset=['Driver'])
    
    # 计算总积分
    total_points = full_season.groupby('Driver')['Points'].sum().reset_index()
    total_points.columns = ['Driver', 'TotalPoints']
    
    driver_info = driver_info.merge(total_points, on='Driver')
    
    # 按积分排序
    driver_info = driver_info.sort_values('TotalPoints', ascending=False).reset_index(drop=True)
    driver_info['CurrentRank'] = range(1, len(driver_info) + 1)
    
    # 保存
    driver_info.to_csv('./data/drivers_info.csv', index=False)
    print("\n✅ 车手信息已保存到 drivers_info.csv")
    
    return driver_info

In [6]:
def extract_team_info(full_season):
    """
    从比赛数据中提取车队信息
    """
    # 计算车队总积分
    team_points = full_season.groupby('Team')['Points'].sum().reset_index()
    team_points.columns = ['Team', 'TotalPoints']
    team_points = team_points.sort_values('TotalPoints', ascending=False).reset_index(drop=True)
    team_points['CurrentRank'] = range(1, len(team_points) + 1)
    
    # 手动添加车队层级（根据积分）
    def assign_tier(rank):
        if rank <= 3:
            return 0
        elif rank <= 5:
            return 1
        else:
            return 2
    
    team_points['Tier'] = team_points['CurrentRank'].apply(assign_tier)
    
    # 保存
    team_points.to_csv('./data/teams_info.csv', index=False)
    print("✅ 车队信息已保存到 teams_info.csv")
    
    return team_points


def calculate_driver_features(full_season):
    """
    计算每个车手的特征
    """
    # ✅ 检查并添加 TrackType 如果不存在
    if 'TrackType' not in full_season.columns:
        print("⚠️ TrackType列不存在，正在添加...")
        track_types = {
            'Australia': 'balanced', 'China': 'high_speed', 'Japan': 'technical',
            'Bahrain': 'high_speed', 'Saudi Arabia': 'high_speed', 'Miami': 'high_speed',
            'Emilia Romagna': 'technical', 'Monaco': 'technical', 'Spain': 'balanced',
            'Canada': 'balanced', 'Austria': 'technical', 'Great Britain': 'balanced',
            'Belgium': 'high_speed', 'Hungary': 'technical', 'Netherlands': 'technical',
            'Italy': 'high_speed', 'Azerbaijan': 'high_speed', 'Singapore': 'technical',
            'United States': 'balanced', 'Mexico': 'balanced', 'Brazil': 'balanced',
            'Las Vegas': 'high_speed', 'Qatar': 'high_speed', 'Abu Dhabi': 'balanced',
        }
        full_season['TrackType'] = full_season['Race'].map(track_types)
    
    drivers = full_season['Driver'].unique()
    features = []
    
    for driver in drivers:
        driver_data = full_season[full_season['Driver'] == driver].copy()
        
        # 基本信息
        team = driver_data['Team'].iloc[-1]
        total_points = driver_data['Points'].sum()
        
        # 完赛率
        total_races = len(driver_data)
        finished_races = len(driver_data[driver_data['Status'] == 'Finished'])
        dnf_rate = 1 - (finished_races / total_races) if total_races > 0 else 0
        
        # 平均排名（只算完赛的）
        finished_data = driver_data[driver_data['Status'] == 'Finished']
        avg_position = finished_data['Position'].mean() if len(finished_data) > 0 else 20
        
        # 最近5场表现
        recent_5 = driver_data.tail(5)
        recent_5_finished = recent_5[recent_5['Status'] == 'Finished']
        recent_avg = recent_5_finished['Position'].mean() if len(recent_5_finished) > 0 else avg_position
        
        # 赛道类型表现
        track_perfs = {}
        for track_type in ['high_speed', 'balanced', 'technical']:
            type_data = driver_data[driver_data['TrackType'] == track_type]
            type_finished = type_data[type_data['Status'] == 'Finished']
            track_perfs[f'{track_type}_avg'] = type_finished['Position'].mean() if len(type_finished) > 0 else avg_position
            track_perfs[f'{track_type}_races'] = len(type_finished)
        
        features.append({
            'Driver': driver,
            'Team': team,
            'TotalPoints': total_points,
            'TotalRaces': total_races,
            'FinishedRaces': finished_races,
            'DNFRate': dnf_rate,
            'AvgPosition': avg_position,
            'Recent5Avg': recent_avg,
            'HighSpeedAvg': track_perfs['high_speed_avg'],
            'HighSpeedRaces': track_perfs['high_speed_races'],
            'BalancedAvg': track_perfs['balanced_avg'],
            'BalancedRaces': track_perfs['balanced_races'],
            'TechnicalAvg': track_perfs['technical_avg'],
            'TechnicalRaces': track_perfs['technical_races'],
        })
    
    features_df = pd.DataFrame(features)
    features_df = features_df.sort_values('TotalPoints', ascending=False).reset_index(drop=True)
    
    # 保存
    features_df.to_csv('./data/driver_features.csv', index=False)
    print("✅ 车手特征已保存到 driver_features.csv")
    
    return features_df

In [ ]:
# =================== 主流程 ===================

# 1. 收集数据（包括 2024 和 2025）
full_season = collect_race_data(
    years=[2025],  # 🔑 包含两个赛季
    force_refresh=False  # 设为 True 强制重新下载
)

if full_season is not None:
    # 2. 提取车队信息
    teams_info = extract_team_info(full_season)
    
    # 3. 提取车手信息
    drivers_info = extract_driver_info(full_season)
    
    # 4. 计算车手特征
    driver_features = calculate_driver_features(full_season)
    
    print("\n✅ 所有数据准备完成！")
    print(f"   - 总记录数: {len(full_season)}")
    print(f"   - 包含赛季: {full_season['Season'].unique()}")
    print(f"   - QualifyingPosition 列存在: {'QualifyingPosition' in full_season.columns}")
else:
    print("❌ 数据收集失败")

In [8]:
import os

def collect_race_data_manual(force_refresh=False):
    """
    使用手动指定的比赛列表，避免依赖schedule API
    """
    csv_file = './data/f1_multi_season_results.csv'
    
    # 检查CSV是否已存在
    if os.path.exists(csv_file) and not force_refresh:
        print(f"✅ Found existing data file: {csv_file}")
        print("Loading...")
        full_data = pd.read_csv(csv_file)
        
        # ✅ 如果CSV中没有TrackType列，添加它
        if 'TrackType' not in full_data.columns:
            print("⚠️ Adding TrackType to existing data...")
            full_data['TrackType'] = full_data['Race'].map(track_types)
            full_data.to_csv(csv_file, index=False)
            print("✅ TrackType added and saved")
        
        print(f"Loaded {len(full_data)} records")
        return full_data
    
    # 2024赛季所有比赛
    races_2024 = [
        ('Bahrain', 1, '2024-03-02'),
        ('Saudi Arabia', 2, '2024-03-09'),
        ('Australia', 3, '2024-03-24'),
        ('Japan', 4, '2024-04-07'),
        ('China', 5, '2024-04-21'),
        ('Miami', 6, '2024-05-05'),
        ('Emilia Romagna', 7, '2024-05-19'),
        ('Monaco', 8, '2024-05-26'),
        ('Canada', 9, '2024-06-09'),
        ('Spain', 10, '2024-06-23'),
        ('Austria', 11, '2024-06-30'),
        ('Great Britain', 12, '2024-07-07'),
        ('Hungary', 13, '2024-07-21'),
        ('Belgium', 14, '2024-07-28'),
        ('Netherlands', 15, '2024-08-25'),
        ('Italy', 16, '2024-09-01'),
        ('Azerbaijan', 17, '2024-09-15'),
        ('Singapore', 18, '2024-09-22'),
        ('United States', 19, '2024-10-20'),
        ('Mexico', 20, '2024-10-27'),
        ('Brazil', 21, '2024-11-03'),
        ('Las Vegas', 22, '2024-11-23'),
        ('Qatar', 23, '2024-12-01'),
        ('Abu Dhabi', 24, '2024-12-08'),
    ]
    
    # 2025赛季已完成的比赛
    races_2025 = [
        ('Australia', 1, '2025-03-16'),
        ('China', 2, '2025-03-23'),
        ('Japan', 3, '2025-04-06'),
        ('Bahrain', 4, '2025-04-13'),
        ('Saudi Arabia', 5, '2025-04-20'),
        ('Miami', 6, '2025-05-04'),
        ('Emilia Romagna', 7, '2025-05-18'),
        ('Monaco', 8, '2025-05-25'),
        ('Spain', 9, '2025-06-01'),
        ('Canada', 10, '2025-06-15'),
        ('Austria', 11, '2025-06-29'),
        ('Great Britain', 12, '2025-07-06'),
        ('Belgium', 13, '2025-07-27'),
        ('Hungary', 14, '2025-08-03'),
        ('Netherlands', 15, '2025-08-31'),
        ('Italy', 16, '2025-09-07'),
        ('Azerbaijan', 17, '2025-09-21'),
        ('Singapore', 18, '2025-10-05'),
        ('United States', 19, '2025-10-19'),
        ('Mexico', 20, '2025-10-26'),
        ('Brazil', 21, '2025-11-09'),
        ('Las Vegas', 22, '2025-11-22'),
        ('Qatar', 23, '2025-11-30'),
    ]
    
    all_races = []
    
    # # 处理2024赛季
    # print(f"\n{'='*60}")
    # print(f"COLLECTING 2024 SEASON")
    # print(f"{'='*60}")
    # for race_name, round_num, date in races_2024:
    #     race_data = fetch_race_results(2024, race_name, round_num, date)
    #     if race_data is not None:
    #         race_data['Season'] = 2024
    #         all_races.append(race_data)
    
    # 处理2025赛季
    print(f"\n{'='*60}")
    print(f"COLLECTING 2025 SEASON")
    print(f"{'='*60}")
    for race_name, round_num, date in races_2025:
        race_data = fetch_race_results(2025, race_name, round_num, date)
        if race_data is not None:
            race_data['Season'] = 2025
            all_races.append(race_data)
    
    # 合并数据
    if all_races:
        full_data = pd.concat(all_races, ignore_index=True)
        full_data = full_data.sort_values(['Season', 'Round'])
        full_data['GlobalRound'] = full_data.groupby(['Season', 'Round']).ngroup() + 1
        
        # ✅ 确保 TrackType 列存在
        if 'TrackType' not in full_data.columns:
            full_data['TrackType'] = full_data['Race'].map(track_types)
            print("✅ TrackType column added")
        
        full_data.to_csv(csv_file, index=False)
        
        print(f"\n{'='*60}")
        print(f"DATA COLLECTION COMPLETE")
        print(f"{'='*60}")
        print(f"✓ Total records: {len(full_data)}")
        print(f"✓ Total races: {full_data.groupby(['Season', 'Round']).ngroups}")
        print(f"✓ TrackType column: {'✓' if 'TrackType' in full_data.columns else '✗'}")
        
        # 显示TrackType分布
        if 'TrackType' in full_data.columns:
            print(f"\n赛道类型分布:")
            print(full_data.groupby('TrackType')['Round'].nunique())
        
        return full_data
    else:
        print("❌ No data collected")
        return None

In [16]:
full_season = collect_race_data_manual()


✅ Found existing data file: ./data/f1_multi_season_results.csv
Loading...
Loaded 479 records


In [17]:
driver_info = extract_driver_info(full_season)
print("\n  TOP 10 driver ：")
print(driver_info.head(10))


✅ 车手信息已保存到 drivers_info.csv

  TOP 10 driver ：
         Driver DriverCode             Team  TotalPoints  CurrentRank
0      L NORRIS        NOR          McLaren        394.0            1
1  M VERSTAPPEN        VER  Red Bull Racing        382.0            2
2     O PIASTRI        PIA          McLaren        375.0            3
3     G RUSSELL        RUS         Mercedes        304.0            4
4     C LECLERC        LEC          Ferrari        221.0            5
5    L HAMILTON        HAM          Ferrari        135.0            6
6   K ANTONELLI        ANT         Mercedes        133.0            7
7       A ALBON        ALB         Williams         70.0            8
8       C SAINZ        SAI         Williams         55.0            9
9      I HADJAR        HAD     Racing Bulls         50.0           10


In [18]:
print("\n" + "="*50)
team_info = extract_team_info(full_season)
print("\n team point:")
print(team_info)


✅ 车队信息已保存到 teams_info.csv

 team point:
              Team  TotalPoints  CurrentRank  Tier
0          McLaren        769.0            1     0
1         Mercedes        449.0            2     0
2  Red Bull Racing        403.0            3     0
3          Ferrari        356.0            4     1
4         Williams        125.0            5     1
5     Racing Bulls         88.0            6     2
6     Aston Martin         77.0            7     2
7     Haas F1 Team         69.0            8     2
8      Kick Sauber         68.0            9     2
9           Alpine         20.0           10     2


In [19]:
# 5. 计算车手特征
print("\n" + "="*50)
driver_features = calculate_driver_features(full_season)
print("\n driver features TOP 10：")
print(driver_features[['Driver', 'TotalPoints', 'AvgPosition', 'Recent5Avg', 
                       'HighSpeedAvg', 'BalancedAvg', 'DNFRate']].head(10))


✅ 车手特征已保存到 driver_features.csv

 driver features TOP 10：
         Driver  TotalPoints  AvgPosition  Recent5Avg  HighSpeedAvg  \
0      L NORRIS        394.0     2.238095        2.00      2.238095   
1  M VERSTAPPEN        382.0     3.086957        1.80      3.086957   
2     O PIASTRI        375.0     2.863636        4.25      2.863636   
3     G RUSSELL        304.0     4.086957        5.00      4.086957   
4     C LECLERC        221.0     5.142857        4.25      5.142857   
5    L HAMILTON        135.0     6.750000        8.00      6.750000   
6   K ANTONELLI        133.0     7.176471        5.80      7.176471   
7       A ALBON         70.0     8.466667       12.00      8.466667   
8       C SAINZ         55.0    10.071429        7.00     10.071429   
9      I HADJAR         50.0     9.384615       10.00      9.384615   

   BalancedAvg   DNFRate  
0     2.238095  0.125000  
1     3.086957  0.041667  
2     2.863636  0.083333  
3     4.086957  0.041667  
4     5.142857  0.125000 